# Exact multiplication with native FlagQuantum

Preserve both original tutorial examples and their order. Use coherent quantum values, local CPU execution and exact named results; seeded local samples replace the upstream cloud sampler. No hardware result is claimed.


In [ ]:
from collections import Counter
from fractions import Fraction
from random import Random
from typing import Annotated

import flagquantum as fq
from flagquantum import synth
from flagquantum.synth import Output, QNum, allocate, hadamard_transform, qfunc, synthesize


## Multiply words

The original values and input domains are preserved. Any signed/fractional values below use exact numeric readout rather than manual bit-string decoding.


In [ ]:
@qfunc
def multiply_words(a: Output[QNum], b: Output[QNum], res: Output[QNum]) -> None:
    a |= 4
    b |= 5
    res |= a * b


In [ ]:
multiply_words_program = synthesize(multiply_words)
multiply_words_result = fq.run(multiply_words_program)
multiply_words_probabilities = dict(multiply_words_program.probabilities(multiply_words_result, registers=('a', 'b', 'res')))
expected = {(4, 5, 20): 1.0}
assert multiply_words_probabilities.keys() == expected.keys()
assert all(abs(multiply_words_probabilities[key]-value) < 1e-12 for key,value in expected.items())
multiply_words_counts = Counter(Random(17).choices(list(multiply_words_probabilities), weights=list(multiply_words_probabilities.values()), k=1000))
assert sum(multiply_words_counts.values()) == 1000
print("Probabilities:", multiply_words_probabilities)
print("Counts:", dict(multiply_words_counts))
print("Resources:", multiply_words_program.report["resources"])


## Multiply scaled

The original values and input domains are preserved. Any signed/fractional values below use exact numeric readout rather than manual bit-string decoding.


In [ ]:
@qfunc
def multiply_scaled(a: Output[Annotated[QNum, 2]], res: Output[QNum]) -> None:
    allocate(a)
    hadamard_transform(a)
    res |= 3.5 * a


In [ ]:
multiply_scaled_program = synthesize(multiply_scaled)
multiply_scaled_result = fq.run(multiply_scaled_program)
multiply_scaled_probabilities = dict(multiply_scaled_program.numeric_probabilities(multiply_scaled_result, registers=('a', 'res')))
expected = {(a, Fraction(7, 2)*a): 1/4 for a in range(4)}
assert multiply_scaled_probabilities.keys() == expected.keys()
assert all(abs(multiply_scaled_probabilities[key]-value) < 1e-12 for key,value in expected.items())
multiply_scaled_counts = Counter(Random(17).choices(list(multiply_scaled_probabilities), weights=list(multiply_scaled_probabilities.values()), k=1000))
assert sum(multiply_scaled_counts.values()) == 1000
print("Probabilities:", multiply_scaled_probabilities)
print("Counts:", dict(multiply_scaled_counts))
print("Resources:", multiply_scaled_program.report["resources"])


## What is checked

These examples keep their original inputs, including exact binary coefficient 3.5 where present. Independent Circuit acceptance must additionally check the complete complex state and work cleanup; probabilities and samples alone do not certify phase.
